# Lab Instructions — Real-Time Fraud Detection Pipeline
## Your Step-by-Step Source of Truth

This notebook contains **every instruction** you need to complete the Real-Time Fraud Detection Pipeline lab. Follow each phase in order. Every step tells you **exactly which notebook to open**, **which cell to run**, and **what output to expect**.

> **⚠️ Important:** This notebook is your **guide only** — you do NOT run any code here. All code runs in the three companion notebooks listed below.

---

### This Is a Hands-On Lab

The main pipeline notebook contains **5 coding exercises** where key parts of the code have been replaced with `<FILL_IN>` placeholders. Your job is to figure out the correct code and replace each placeholder to make the pipeline work end-to-end.

| Exercise | Concept | Where |
|---|---|---|
| 1 | Auto Loader options & streaming trigger | Phase 2 — Bronze Ingestion |
| 2 | PySpark feature engineering | Phase 5 — Silver Enrichment |
| 3 | Watermarks & windowed aggregations | Phase 5 — Velocity Checks |
| 4 | `ai_query()` SQL function | Phase 7 — AI Scoring (endpoint) |
| 5 | MLflow Spark UDF | Phase 7 — AI Scoring (UDF) |

> **Stuck?** Full answers for every exercise are at the **bottom of this notebook** in the Answer Key section.

---

### The Three Notebooks

You will work across three notebooks during this lab. Each one has a specific role:

| # | Notebook Name | Role | When You Use It |
|---|---|---|---|
| 1 | **Kinesis Event Producer Simulator** | Generates fake transaction data files (simulates a live data feed) | Start first, keep running in a separate browser tab |
| 2 | **Real-Time Fraud Detection Pipeline** | Ingests data, builds Bronze/Silver tables, scores fraud — **contains the 5 exercises** | Main working notebook — most of your time is here |
| 3 | **Fraud Detection Model Setup** | Trains a fraud-detection model and deploys it as a web service | Run once before Phase 7 |

### How to Find Cells in Each Notebook

Every code cell in the three notebooks has a **title** displayed above it (in bold text at the top of the cell). This guide refers to cells by their exact title name. For example, when you see:

> **Run the cell titled "Continuous producer loop (10s interval)"**

...scroll through the notebook until you find a cell with that exact title at the top. Then click the **▶ Run** button on that cell.

### How to Run a Cell

1. Click anywhere inside the cell to select it (a blue border appears)
2. Click the **▶ Run** button in the top-right corner of the cell, **OR** press **Shift + Enter** on your keyboard
3. Wait for the cell to finish — you'll see output appear below it and the spinner will stop
4. **Do not skip cells** — run them in the order listed in these instructions

## Prerequisites — Before You Start

Before beginning this lab, make sure you have the following:

1. **Databricks Workspace Access** — You must be logged into a Databricks workspace
2. **Compute Available** — Serverless compute will be used automatically. No cluster setup is needed. **You must ensure your serverless environment is version 5 or later**
3. **Three Browser Tabs Ready** — You will have up to 3 notebooks open at the same time. Open each one in a **new browser tab** (right-click the notebook name → "Open in new tab")

### Finding the Notebooks

In your Databricks workspace:
1. Click **Workspace** in the left sidebar
2. Navigate to `Labs/06 - Real-Time Fraud Detection Pipeline/`
3. You should see four notebooks (including this one):
   * `Lab Instructions - Step by Step Guide` ← **You are here**
   * `Kinesis Event Producer Simulator`
   * `Real-Time Fraud Detection Pipeline` ← **Contains the 5 exercises**
   * `Fraud Detection Model Setup`

If you cannot find these notebooks, ask your instructor for help.

---

## Phase 1 — Start the Event Producer

**Goal:** Start generating fake transaction data so the main pipeline has something to process.

**Which notebook:** `Kinesis Event Producer Simulator`

---

### Step 1.1 — Open the Producer Notebook

1. Go to your Databricks workspace sidebar and click **Workspace**
2. Navigate to `Labs/06 - Real-Time Fraud Detection Pipeline/`
3. **Right-click** on `Kinesis Event Producer Simulator` and select **Open in new tab**
4. You should now have this instructions notebook in one tab and the Producer notebook in another

### Step 1.2 — Run the Reset Cell

1. In the Producer notebook, find the cell titled **"Reset - Clean up previous run"**
   * This is the **2nd cell** in the notebook (right below the intro text)
2. Click inside the cell and press **Shift + Enter** to run it
3. **What you should see:** Messages saying resources were cleaned up or skipped. You may see `ℹ️ Serving endpoint not found (skipped)` — **this is completely normal** for a first run

### Step 1.3 — Create the Catalog and Volume

1. Find the cell titled **"Create Catalog, Schema, and Volume"**
   * This is the **3rd cell**, directly below the reset cell
2. Run it (**Shift + Enter**)
3. **What you should see:** Five green checkmarks (✅) confirming the catalog, schema, volume, landing zone, and bronze table were created:
   ```
   ✅ Catalog  : fintech_demo
   ✅ Schema   : fintech_demo.bronze_layer
   ✅ Volume   : /Volumes/fintech_demo/bronze_layer/kinesis_landing
   ✅ Landing  : /Volumes/fintech_demo/bronze_layer/kinesis_landing/events
   ✅ Bronze   : fintech_demo.bronze_layer.txn_events_bronze
   ```

### Step 1.4 — Configure the Landing Zone Directory

1. Find the cell titled **"Confirm Volume Creation"**
   * This is the **4th cell**, directly below the previous one
2. Run it (**Shift + Enter**)
3. **What you should see:**
   ```
   ✅ Producer configured
      Landing zone: /Volumes/fintech_demo/bronze_layer/kinesis_landing/events
   ```

### Step 1.5 — Load the Event Generator

1. Find the cell titled **"Define the Kinesis event generator"**
   * This is the **5th cell**
2. Run it (**Shift + Enter**)
3. **What you should see:** No visible output — this cell just defines helper functions. **If it finishes without an error, you're good.** Move on to the next step

### Step 1.6 — Start the Continuous Producer

1. Find the cell titled **"Continuous producer loop (10s interval)"**
   * This is the **7th cell**, below the "Start Continuous Event Production" markdown header
2. Run it (**Shift + Enter**)
3. **What you should see:** Lines appearing every 10 seconds showing new files being created:
   ```
   🚀 Starting continuous event production...
      Dropping 50-event files every 10 seconds to: ...

      [14:32:10]  📥 File 1: events_20260401_143210.json  (running total: 50 events)
      [14:32:20]  📥 File 2: events_20260401_143220.json  (running total: 100 events)
   ```
4. **⚠️ CRITICAL: Do NOT stop this cell!** Leave it running. It will keep producing data in the background
5. **Switch back to this instructions tab** and continue to Phase 2

> **Troubleshooting:** If the cell immediately finishes with no output, something went wrong in a previous step. Go back and re-run Steps 1.2 through 1.5 in order.

---

> ✅ **Checkpoint:** The Producer is running and continuously creating JSON event files. You are ready for Phase 2.

## Phase 2 — Bronze Ingestion (Consumer Pipeline)

**Goal:** Use Auto Loader to read the JSON files created by the Producer and store them in a Bronze Delta table.

**Which notebook:** `Real-Time Fraud Detection Pipeline`

---

### Step 2.1 — Open the Consumer Notebook

1. Go back to your workspace file browser
2. **Right-click** on `Real-Time Fraud Detection Pipeline` and select **Open in new tab**
3. You should now have **three browser tabs** open:
   * **Tab 1:** This instructions notebook
   * **Tab 2:** The Producer notebook (still running!)
   * **Tab 3:** The Consumer notebook (you just opened this)

### Step 2.2 — Set Configuration Variables

1. In the Consumer notebook, find the cell titled **"Create catalog, schema, and landing volume"**
   * This is the **3rd cell** (right below the "Environment Setup" markdown header)
2. Run it (**Shift + Enter**)
3. **What you should see:** Five green checkmarks (✅) showing the configuration variables — identical to what the Producer printed

### Step 2.3 — Ensure the Landing Zone Exists

1. Find the cell titled **"Ensure Landing Zone Directory Exists"**
   * This is the **4th cell**, right below the previous one
2. Run it (**Shift + Enter**)
3. **What you should see:**
   ```
   ✅ Producer configured
      Landing zone: /Volumes/fintech_demo/bronze_layer/kinesis_landing/events
   ```

### Step 2.4 — Peek at the Landing Zone

1. Find the cell titled **"Peek at the landing zone"**
   * This is the **6th cell**, below the "Peek at the Landing Zone" markdown header
2. Run it (**Shift + Enter**)
3. **What you should see:** A list of JSON files that the Producer has created so far:
   ```
   📂 Landing zone: /Volumes/fintech_demo/bronze_layer/kinesis_landing/events
      Total JSON files: 5

      • events_20260401_143210.json  (8,320 bytes)
      • events_20260401_143220.json  (8,415 bytes)
      ...
   ```
4. **If you see "⚠️ No files yet!"** → switch to the Producer tab and make sure the producer loop cell is still running (7th cell). Wait 20 seconds, then come back and re-run this cell

### Step 2.5 — 🧪 Exercise 1: Complete the Auto Loader Cell

> **This is your first exercise!** The code has `<FILL_IN>` placeholders you must replace.

1. Find the cell titled **"Auto Loader: Ingest JSON into Bronze Delta table"**
   * This is the **8th cell**, below the "Ingest into Bronze with Auto Loader" markdown header
2. **Read the markdown cell above it** — it explains exactly what Auto Loader options are needed
3. **Replace each `<FILL_IN>`** with the correct value:
   * `cloudFiles.format` → Think about what file type the Producer is creating
   * `cloudFiles.schemaLocation` → A variable defined in Step 2.2 holds the path
   * `cloudFiles.inferColumnTypes` → A string boolean
   * `cloudFiles.schemaEvolutionMode` → The markdown cell above mentions this mode by name
   * `.trigger(...)` → The markdown cell explains which trigger is serverless-compatible
4. Run the cell (**Shift + Enter**)
5. **What you should see when correct:**
   ```
   ✅ Auto Loader stream completed
      Bronze table: fintech_demo.bronze_layer.txn_events_bronze
   ```
6. **If you get a SyntaxError:** You still have an unreplaced `<FILL_IN>` — check your code carefully
7. **If you're stuck:** Check the Answer Key at the bottom of this instructions notebook

### Step 2.6 — Verify the Bronze Table

1. Find the cell titled **"Inspect Bronze table schema and row count"**
   * This is the **10th cell**, below the "Verify the Bronze Table" markdown header
2. Run it (**Shift + Enter**)
3. **What you should see:** A row count and the table schema:
   ```
   📊 Row count: 250

   📝 Schema:
   root
    |-- event_id: string
    |-- event_type: string
    |-- timestamp: string
    |-- amount: double
    ...
   ```
4. **The exact row count will vary** — it depends on how many files the Producer created before you ran Auto Loader. Any number above 0 means it's working!

### Step 2.7 — Preview the Bronze Data

1. Find the cell titled **"Preview Bronze table data"**
   * This is the **11th cell**, right below the previous one
2. Run it (**Shift + Enter**)
3. **What you should see:** A data table showing transaction records with columns like `event_id`, `event_type`, `timestamp`, `amount`, `currency`, `sender_id`, `receiver_id`, etc.
4. Scroll through the data to get a feel for what the events look like

---

> ✅ **Checkpoint:** You have successfully ingested JSON files into a Bronze Delta table using Auto Loader. The Producer is still running and creating new files in the background.

## Phase 3 — Incremental Pickup Demo

**Goal:** Demonstrate that Auto Loader only processes NEW files — it never re-reads files it has already ingested.

**Which notebook:** `Real-Time Fraud Detection Pipeline` (stay in the same tab)

---

### Step 3.1 — Note the Current Row Count

1. Look at the output of the cell you ran in Step 2.6 (titled **"Inspect Bronze table schema and row count"**)
2. **Write down the row count** on a piece of paper or in a text file (e.g., "250"). You'll compare it in a moment

### Step 3.2 — Wait for New Files

1. Wait **30 to 60 seconds** — the Producer notebook is still running and dropping new JSON files every 10 seconds
2. (Optional) Switch to the Producer tab briefly to confirm it's still printing new file lines, then switch back

### Step 3.3 — Re-Run Auto Loader

1. Go back to the Consumer notebook
2. Find the cell titled **"Auto Loader: Ingest JSON into Bronze Delta table"** (this is the same cell you ran in Step 2.5)
3. Run it again (**Shift + Enter**)
4. **What happens behind the scenes:** Auto Loader uses its checkpoint to skip files it already processed and only ingests the NEW files that arrived while you were waiting

### Step 3.4 — Verify the Count Increased

1. Re-run the cell titled **"Inspect Bronze table schema and row count"** (the same cell from Step 2.6)
2. **What you should see:** The row count is now **HIGHER** than before. For example, if it was 250 before and 60 seconds passed (6 new files × 50 events = 300 new events), you might now see \~550
3. **The key takeaway:** Auto Loader only processed new files — no duplicates, no reprocessing. This is exactly-once ingestion

---

> ✅ **Checkpoint:** You have proven that Auto Loader supports incremental ingestion. It tracks which files have been processed using a checkpoint, so it never duplicates data.

## Phase 4 — Schema Evolution Demo

**Goal:** Show that when upstream data adds a new field, Auto Loader automatically detects and merges it into the table schema — no manual DDL required.

**Which notebooks:** First the `Kinesis Event Producer Simulator`, then back to `Real-Time Fraud Detection Pipeline`

---

### Step 4.1 — Stop the Current Producer

1. **Switch to the Producer notebook tab** (the one that's been running the continuous loop)
2. Find the cell titled **"Continuous producer loop (10s interval)"** — it should still be running (you'll see a spinner or active indicator)
3. Click the **■ Stop** button on that cell to interrupt it
   * The Stop button is in the top-right corner of the cell (it replaces the Run button while a cell is executing)
4. **What you should see:** A message like:
   ```
   ⏹️  Producer stopped after 12 files (600 events)
   ```

### Step 4.2 — Start the Schema Evolution Producer

1. In the same Producer notebook, scroll down to the **bottom** of the notebook
2. Find the cell titled **"Schema evolution producer (adds risk_score)"**
   * This is the **9th cell** in the Producer notebook, below the "Schema Evolution Mode" markdown header
3. Run it (**Shift + Enter**)
4. **What you should see:** Lines appearing every 10 seconds, similar to before, but now the message says events include a new `risk_score` field:
   ```
   🚀 Starting schema-evolution event production...
      Events now include 'risk_score' field (0.0 – 1.0)
      Dropping files every 10 seconds to: ...
   ```
5. Let it run for **at least 20–30 seconds** (so 2–3 files are created), then you can either leave it running or stop it — either is fine for this demo

### Step 4.3 — Re-Run Auto Loader to Pick Up the New Schema

1. **Switch back to the Consumer notebook tab**
2. Find the cell titled **"Auto Loader: Ingest JSON into Bronze Delta table"** (same cell as before)
3. Run it (**Shift + Enter**)
4. **What happens behind the scenes:** Auto Loader detects that new files contain a `risk_score` field that wasn't in the original schema. Because we configured `addNewColumns` mode, it automatically adds the column. However **you will see an error** this is ok.
5. Re-run the cell again. This time it successfully completes and the data for the new column has been added.

### Step 4.4 — Verify the New Column

1. Re-run the cell titled **"Inspect Bronze table schema and row count"**
2. **What you should see:** In the schema printout, a new `risk_score` column now appears at the bottom:
   ```
   📝 Schema:
   root
    |-- event_id: string
    |-- event_type: string
    ...
    |-- risk_score: double    ← NEW! This was added automatically
   ```
3. The row count will also be higher than before (the new files with `risk_score` were ingested)

### Step 4.5 — Stop the Schema Evolution Producer (if still running)

1. **Switch to the Producer notebook tab**
2. If the schema evolution cell is still running, click the **■ Stop** button to stop it
3. You are done with the Producer notebook for the rest of this lab

---

> ✅ **Checkpoint:** You have demonstrated schema evolution. Auto Loader detected a new field (`risk_score`) in incoming data and automatically merged it into the Bronze table — zero downtime, zero manual DDL.

## Phase 5 — Silver Layer: Enrichment & Velocity Checks

**Goal:** Transform raw Bronze data into curated Silver tables with fraud-detection features and velocity aggregations.

**Which notebook:** `Real-Time Fraud Detection Pipeline`

---

### Step 5.1 — Set Up Silver Configuration

1. In the Consumer notebook, scroll down past the Bronze cells to find the cell titled **"Silver layer configuration"**
   * This is the **13th cell**, It's located below the "Silver Layer" markdown header
2. Run it (**Shift + Enter**)
3. **What you should see:**
   ```
   ✅ Silver schema  : fintech_demo.silver_layer
   ✅ Silver table   : fintech_demo.silver_layer.txn_events_silver
   ✅ Velocity table : fintech_demo.silver_layer.txn_velocity_1min
   ```

### Step 5.2 — 🧪 Exercise 2: Complete the Silver Enrichment Stream

> **Your second exercise!** You need to add four derived columns that serve as fraud signals.

1. Find the cell titled **"Silver enrichment stream (Bronze to Silver)"**
   * This is the **14th cell**, after the Silver configuration cell
2. **Read the markdown cell above it** — it explains the Silver layer’s purpose and the columns you need
3. **Replace each `<FILL_IN>`** with the correct PySpark expression:
   * `hour_of_day` → You need a function that extracts the hour from a timestamp (look at the imports!)
   * `day_of_week` → Similar — a function that extracts the day of the week
   * `is_high_value` → A conditional expression: True when `amount > 5000`, otherwise False
   * `is_off_hours` → A condition combining two checks with `|` (OR): hour < 6 or hour > 22
4. Run the cell (**Shift + Enter**)
5. **What you should see when correct:**
   ```
   ✅ Silver enrichment stream completed
      Silver table: fintech_demo.silver_layer.txn_events_silver
      Row count:    600
   ```
   (Your row count will vary based on how many events were in Bronze)
6. **If you're stuck:** The functions you need are all in the `import` statement at the top of the cell. Check the Answer Key if needed.

### Step 5.3 — Verify the Silver Table

1. Find the cell titled **"Verify Silver enriched table"**
   * This is the **15th cell**, it's the next code cell from the previous one ran
2. Run it (**Shift + Enter**)
3. **What you should see:**
   * The schema printed out, showing **new derived columns** like `hour_of_day`, `day_of_week`, `is_high_value`, `is_off_hours`
   * A data table showing 10 sample rows with all the enriched columns
4. **Take a moment to examine the data** — notice how each transaction now has fraud-relevant features that were computed from the raw Bronze data

### Step 5.4 — 🧪 Exercise 3: Complete the Velocity Aggregation Stream

> **Your third exercise!** You need to set a watermark, define a tumbling window, and pick the right output mode.

1. Find the cell titled **"Velocity aggregation stream (10-min windows)"**
   * This is the **17th cell**, it's located below the "Velocity Checks" markdown header
2. **Read the markdown cell above it** — it explains watermarks, tumbling windows, and output modes
3. **Replace each `<FILL_IN>`** with the correct value:
   * `.withWatermark(...)` → Takes the column name (a string) and a lateness threshold (e.g., `"1 minute"`)
   * `window(...)` → Takes the timestamp column and the window size (e.g., `"1 minute"`)
   * `.outputMode(...)` → The markdown cell above explains which mode emits results only after the watermark passes a window’s end
4. Run the cell (**Shift + Enter**)
5. **What you should see when correct:**
   ```
   ✅ Velocity aggregation stream completed
      Velocity table: fintech_demo.silver_layer.txn_velocity_1min
      Window count:   150
   ```

### Step 5.5 — Review Velocity Results

1. Find the cell titled **"Review velocity check results"**
   * This is the **18th cell**, it's the next code cell from the previous one ran
2. Run it (**Shift + Enter**)
3. **What you should see:** A table showing senders ranked by how many transactions they made in a 1-minute window. Pay attention to:
   * **`is_velocity_alert = true`** — sender made 2+ transactions in 1 minute (suspicious!)
   * **`is_high_volume = true`** — sender moved more than $15,000 in 1 minute (suspicious!)
4. In a real fraud detection system, these flags would trigger alerts for investigation

---

> ✅ **Checkpoint:** You have built a Silver layer with enriched fraud features and streaming velocity aggregations. The pipeline now transforms raw data into business-ready tables with real fraud signals.

## Phase 6 — Train and Deploy the Fraud Model

**Goal:** Train a machine learning model to score transactions for fraud, register it in Unity Catalog, and deploy it as a live serving endpoint.

**Which notebook:** `Fraud Detection Model Setup`

> **⏱ Time estimate:** This phase takes approximately **5–10 minutes**, mostly waiting for the serving endpoint to provision.

---

### Step 6.1 — Open the Model Setup Notebook

1. Go to your workspace sidebar → click **Workspace** → navigate to `Labs/06 - Real-Time Fraud Detection Pipeline/`
2. **Right-click** on `Fraud Detection Model Setup` and select **Open in new tab**
3. You may now have 4 browser tabs open — that's fine. You can close the Producer tab if it's no longer running

### Step 6.2 — Set Configuration

1. In the Model Setup notebook, find the cell titled **"Configuration and models schema"**
   * This is the **2nd cell** in the notebook (right below the intro text)
2. Run it (**Shift + Enter**)
3. **What you should see:**
   ```
   ✅ Models schema : fintech_demo.models
   ✅ Model name    : fintech_demo.models.fraud_scorer
   ✅ Endpoint name : fintech-fraud-scoring
   ```

### Step 6.3 — Generate Training Data

1. Find the cell titled **"Generate synthetic fraud training data"**
   * This is the **3rd cell**
2. Run it (**Shift + Enter**)
3. **What you should see:** A summary of the training data showing 10,000 rows with columns like `amount`, `hour_of_day`, `day_of_week`, `is_high_value`, `is_off_hours`, plus a `is_fraud` label column
4. You may also see a fraud rate statistic printed (approximately 5% of transactions are labeled as fraud)

### Step 6.4 — Train the Model

1. Find the cell titled **"Train model and register in Unity Catalog"**
   * This is the **4th cell**
2. Run it (**Shift + Enter**)
3. **Wait for it to complete** — this trains a RandomForest classifier, evaluates it, wraps it in a custom PyFunc, and registers it in Unity Catalog. May take **1–2 minutes**
4. **What you should see:**
   * A test AUC-ROC score (should be around **0.85 or higher** — this means the model is good at distinguishing fraud from legitimate transactions)
   * A classification report showing precision and recall
   * A confirmation message:
     ```
     ✅ Model registered: fintech_demo.models.fraud_scorer
     ```

### Step 6.5 — Validate the Model

1. Find the cell titled **"Validate model locally before deployment"**
   * This is the **5th cell**, it's the next code cell
2. Run it (**Shift + Enter**)
3. **What you should see:** Test predictions for 4 sample transactions:
   * A **$50 transaction at 2pm** → 🟢 LOW risk
   * An **$8,500 transaction at 3am** → 🔴 HIGH risk
   * And so on...
4. **Verify the results make intuitive sense** — high-value off-hours transactions should score higher than small daytime ones

### Step 6.6 — Deploy the Serving Endpoint

1. Find the cell titled **"Create model serving endpoint"**
   * This is the **6th cell**, it's the next code cell
2. Run it (**Shift + Enter**)
3. **⚠️ This cell takes 5–10 minutes to complete!** It is provisioning a cloud server to host your model. You will see a progress indicator or status messages while it works
4. **What you should see when it finishes:**
   ```
   ✅ Endpoint 'fintech-fraud-scoring' is READY
   ```
5. **While you wait:** This is a great time to:
   * Review the educational markdown cells in the Consumer notebook (the ones between code cells that explain concepts)
   * Re-read the comparison table in Phase 5 about velocity checks
   * Ask your instructor any questions

### Step 6.7 — Test the Endpoint

1. Find the cell titled **"Test the serving endpoint"**
   * This is the **7th cell** in the Model Setup notebook
2. Run it (**Shift + Enter**)
3. **What you should see:** Live predictions from the deployed endpoint:
   ```
   🌐 Live endpoint test results:
      Suspicious txn ($8,500 at 3am): 0.7832
      Normal txn ($25 at 2pm):        0.0412
   ```
4. The suspicious transaction should have a **much higher** score than the normal one
5. You are done with this notebook — you can leave this tab open or close it

---

> ✅ **Checkpoint:** Your fraud-scoring model is trained, registered in Unity Catalog, and deployed as a live serving endpoint. Switch back to the Consumer notebook for Phase 7.

## Phase 7 — AI-Powered Fraud Scoring

**Goal:** Score Silver table transactions using the deployed model — both via the serving endpoint (`ai_query()`) and via a local Spark UDF.

**Which notebook:** `Real-Time Fraud Detection Pipeline`

---

### Step 7.1 — 🧪 Exercise 4: Score with ai_query() (Endpoint Method)

> **Your fourth exercise!** You need to complete the `ai_query()` SQL function call.

1. **Switch back to the Consumer notebook tab**
2. Scroll down to find the cell titled **"Score Silver transactions with ai_query()"**
   * This is the **21st cell**, it's located below the "With ai_query()" markdown header
3. **Read the markdown cell above it** — it explains how `ai_query()` calls a model serving endpoint
4. **Replace each `<FILL_IN>`** with the correct SQL value:
   * First argument → The endpoint name. Since this is inside an f-string, you can reference the Python variable
   * `returnType` → Think about what SQL data type represents a decimal probability between 0.0 and 1.0
5. Run the cell (**Shift + Enter**)
6. **What you should see when correct:**
   ```
   🤖 Scored 200 transactions via 'fintech-fraud-scoring'
   ```
   Plus a data table showing each transaction with a `fraud_score` column (values between 0.0 and 1.0). **Higher scores = more likely fraud**
7. **If you're stuck:** Check the Answer Key at the bottom of this notebook

### Step 7.2 — 🧪 Exercise 5: Score with Spark UDF (No Endpoint)

> **Your fifth and final exercise!** Load the same model as a Spark UDF and apply it.

This is an **alternative** scoring method that runs the same model directly inside Spark — no network call to the endpoint needed.

**If you get MLflow errors when following the below steps:** ensure your serverless compute is running version 5 or above.

1. Find the cell titled **"UDF-based fraud scoring (no endpoint)"**
   * This is the **23rd cell**, it's located below the "With UDF-Based Scoring (No Endpoint Required)" markdown header
2. **Replace each `<FILL_IN>`** with the correct expression:
   * `fraud_udf = ...` → Use `mlflow.pyfunc` to create a Spark UDF. You need to pass `spark`, `model_uri`, and a `result_type`
   * `fraud_udf(...)` → Build a `struct()` of the 5 feature columns, each cast to double. The column names must match what the model expects: `amount`, `hour_of_day`, `day_of_week`, `is_high_value`, `is_off_hours`
3. Run the cell (**Shift + Enter**)
4. **What you should see when correct:**
   ```
   🤖 Scored 200 transactions via Spark UDF (no endpoint)
   ```
   Plus a table sorted by `fraud_score_udf` descending — highest-risk transactions appear at the top

---

> ✅ **Checkpoint:** You have scored transactions for fraud using two different methods. In production, you would write these scores to a Gold table and trigger alerts on high-risk transactions.

## Phase 8 — Cleanup

**Goal:** Remove all demo resources (catalog, tables, volumes, model, and serving endpoint) so you leave the workspace clean.

**Which notebook:** `Real-Time Fraud Detection Pipeline`

---

### Step 8.1 — Stop the Producer (if still running)

1. **Switch to the Producer notebook tab** (if you still have it open)
2. If any cell is still running (spinner visible), click the **■ Stop** button to stop it
3. You can close this browser tab — you're done with the Producer notebook

### Step 8.2 — Run the Cleanup Cell

1. **Switch to the Consumer notebook tab**
2. Scroll all the way to the bottom and find the cell titled **"Cleanup - Remove demo resources"**
   * It's below the "Cleanup" markdown header
3. **⚠️ Read the cell before running it.** It will:
   * Delete the `fintech-fraud-scoring` serving endpoint
   * Drop the entire `fintech_demo` catalog (which removes ALL tables, schemas, volumes, and models inside it)
4. Run it (**Shift + Enter**)
5. **What you should see:**
   ```
   🗑️  Deleted serving endpoint: fintech-fraud-scoring
   🗑️  Dropped catalog (cascade): fintech_demo
   🧹 All demo resources removed
   ```

---

> ✅ **Lab Complete!** Congratulations — you have successfully built a real-time fraud detection pipeline from end to end.
>
> You covered: decoupled producer/consumer architecture, Auto Loader incremental ingestion, schema evolution, Bronze/Silver medallion layers, streaming velocity checks, ML model training and deployment, and AI-powered fraud scoring.

## Lab Complete!
Congratulations! You've successfully completed the Real-Time Fraud Detection Pipeline lab.

---

### Key Takeaways

| Concept | What You Learned |
|---|---|
| **Decoupled Producer/Consumer** | The producer (Kinesis sim) and consumer (Auto Loader) run independently — just like in production |
| **Auto Loader (`cloudFiles`)** | Incrementally discovers and ingests new files from cloud storage with exactly-once guarantees |
| **Trigger.AvailableNow** | Processes all pending files in one micro-batch, then stops — ideal for serverless and scheduled jobs |
| **Schema Inference** | No manual DDL needed — Auto Loader reads the JSON structure and infers column types |
| **Schema Evolution** | New upstream fields (`risk_score`) are automatically merged into the Bronze table (`addNewColumns` mode) |
| **Silver Enrichment** | Streaming Bronze → Silver: parse timestamps, derive fraud features (`is_high_value`, `is_off_hours`), deduplicate |
| **Windowed Aggregations** | Tumbling windows + watermarks compute per-sender velocity in near real time |
| **Channel Fingerprinting** | Detecting multi-channel anomalies as a fraud signal using `COLLECT_SET` and distinct channel counts |
| **`ai_query()` Scoring** | Call a model serving endpoint from SQL to score every transaction with a fraud probability in real time |
| **Model Serving** | Deploy an MLflow model from Unity Catalog to a scalable REST endpoint with zero infrastructure management |
| **Checkpointing** | Each streaming table gets its own checkpoint — never share checkpoints between streams |

---

### Production Next Steps
1. **Replace the simulated Volume path** with an external Volume pointing to your real Kinesis Firehose S3 prefix
2. **Schedule as a continuous Workflow** — `Trigger.AvailableNow` + continuous job trigger for near-real-time ingestion through Silver
3. **Add a Gold alerting table** — write `ai_query()` scored results to a Gold table, trigger alerts on high-risk transactions
4. **A/B test fraud models** — use endpoint traffic splitting to compare model versions with zero downtime
5. **Enable file event notifications** (`cloudFiles.useManagedFileEvents`) for sub-second file discovery at scale
6. **Add data quality expectations** using Lakeflow Declarative Pipelines (formerly DLT) for production guardrails


---

# Answer Key

> **⚠️ Spoiler alert!** The complete solutions for all 5 exercises are below. Try to solve each exercise on your own first — you'll learn much more that way. Use these answers only when you're truly stuck.

## Exercise 1 — Auto Loader: Ingest JSON into Bronze Delta table

The four `.option()` values and the `.trigger()` call:

```python
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")                       # The Producer writes JSON files
    .option("cloudFiles.schemaLocation", CHECKPOINT)            # Persist inferred schema in the checkpoint directory
    .option("cloudFiles.inferColumnTypes", "true")              # Infer real types (double, string, etc.)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Automatically add new fields from upstream
    .load(LANDING_PATH)
)

# ... audit columns stay the same ...

query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)                                 # Process all pending files, then stop
    .toTable(BRONZE_TABLE)
)
```

**Why these values?**
* `"json"` — The Producer writes newline-delimited JSON files (the same format Kinesis Data Firehose uses)
* `CHECKPOINT` — This variable was defined in the configuration cell; it points to a directory inside the Volume
* `"true"` — Without this, every column would be inferred as STRING
* `"addNewColumns"` — When the schema-evolution producer adds `risk_score`, Auto Loader merges it in automatically
* `availableNow=True` — Processes all available files in one micro-batch then stops; required for serverless compute

## Exercise 2 — Silver enrichment stream (Bronze to Silver)

The four derived fraud-signal columns:

```python
silver_stream = (
    bronze_stream
    .withColumn("event_ts", to_timestamp("timestamp"))
    .withColumn("hour_of_day", hour("event_ts"))                      # hour() extracts 0–23
    .withColumn("day_of_week", dayofweek("event_ts"))                  # dayofweek() returns 1=Sun … 7=Sat
    .withColumn("is_high_value", when(col("amount") > 5000, True).otherwise(False))
    .withColumn("is_off_hours", when(
        (hour("event_ts") < 6) | (hour("event_ts") > 22), True        # Before 6 AM or after 10 PM
    ).otherwise(False))
    .withColumn("merchant", coalesce(col("merchant"), lit("N/A")))
    .drop("timestamp")
    .dropDuplicates(["event_id"])
)
```

**Why these expressions?**
* `hour("event_ts")` — Extracts the hour component (0–23) from the parsed timestamp. Fraud patterns often cluster at unusual hours
* `dayofweek("event_ts")` — Returns the day of the week as an integer (1=Sunday through 7=Saturday). Weekend vs. weekday patterns differ
* `when(col("amount") > 5000, True).otherwise(False)` — A conditional expression (like an IF). Flags transactions over $5,000 as high-value
* `(hour("event_ts") < 6) | (hour("event_ts") > 22)` — The `|` operator means OR. Transactions outside business hours are riskier

## Exercise 3 — Velocity aggregation stream (1-min windows)

The watermark, window, and output mode:

```python
velocity_source = (
    spark.readStream
    .table(BRONZE_TABLE)
    .withColumn("event_ts", to_timestamp("timestamp"))
    .withWatermark("event_ts", "1 minute")                     # Column name, then max lateness
)

velocity_agg = (
    velocity_source
    .groupBy(
        window("event_ts", "1 minute"),                        # Column name, then window duration
        "sender_id"
    )
    .agg( ... )
    # ... fraud flags and select stay the same ...
)

query = (
    velocity_agg.writeStream
    .format("delta")
    .outputMode("append")                                      # Append = emit after watermark passes
    .option("checkpointLocation", VELOCITY_CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(VELOCITY_TABLE)
)
```

**Why these values?**
* `.withWatermark("event_ts", "1 minute")` — Tells Spark: "Data arriving more than 1 minute late can be dropped." This lets Spark know when a window is complete
* `window("event_ts", "1 minute")` — Creates non-overlapping (tumbling) 1-minute time buckets. Each sender’s transactions are grouped into these buckets
* `"append"` — In append mode, results for a window are emitted only **after** the watermark passes the window’s end. This guarantees the window is complete. The other modes (`"complete"` and `"update"`) are not supported with watermarked aggregations writing to Delta

## Exercise 4 — Score Silver transactions with ai_query()

The endpoint name and return type:

```sql
ai_query(
    '{ENDPOINT_NAME}',                          -- Resolves to 'fintech-fraud-scoring' via the f-string
    named_struct(
        'amount',        DOUBLE(amount),
        'hour_of_day',   DOUBLE(hour_of_day),
        'day_of_week',   DOUBLE(day_of_week),
        'is_high_value', DOUBLE(CAST(is_high_value AS INT)),
        'is_off_hours',  DOUBLE(CAST(is_off_hours AS INT))
    ),
    returnType => 'DOUBLE'                      -- A fraud probability is a floating-point number
) AS fraud_score
```

**Why these values?**
* `'{ENDPOINT_NAME}'` — Since the entire SQL string is an f-string, `{ENDPOINT_NAME}` gets replaced with the Python variable’s value (`fintech-fraud-scoring`). The single quotes around it make it a SQL string literal
* `'DOUBLE'` — The model returns a fraud probability between 0.0 and 1.0, which is a floating-point (DOUBLE) value. The `returnType` parameter tells `ai_query()` what type to expect back from the endpoint

## Exercise 5 — UDF-based fraud scoring (no endpoint)

Loading the UDF and calling it:

```python
# Load the model as a Spark UDF
fraud_udf = mlflow.pyfunc.spark_udf(
    spark,
    model_uri=model_uri,
    result_type="double"
)

# Apply the UDF with a struct of the 5 feature columns
scored_udf_df = (
    silver_df
    .withColumn(
        "fraud_score_udf",
        fraud_udf(
            struct(
                col("amount").cast("double").alias("amount"),
                col("hour_of_day").cast("double").alias("hour_of_day"),
                col("day_of_week").cast("double").alias("day_of_week"),
                col("is_high_value").cast("double").alias("is_high_value"),
                col("is_off_hours").cast("double").alias("is_off_hours"),
            )
        ),
    )
    .select( ... )
    .orderBy(col("fraud_score_udf").desc())
    .limit(200)
)
```

**Why this code?**
* `mlflow.pyfunc.spark_udf(spark, model_uri=model_uri, result_type="double")` — This loads the registered MLflow model and wraps it as a Spark UDF. The `spark` session, the model URI (pointing to the Champion alias in Unity Catalog), and the expected return type are all required
* `struct(col("amount").cast("double").alias("amount"), ...)` — The model expects a struct (like a row) with exactly 5 named feature columns, all as doubles. The `.alias()` calls ensure the column names inside the struct match what the model was trained on. The `.cast("double")` ensures the types match (booleans like `is_high_value` get converted to 0.0/1.0)